# Stage 3 — Variational Autoencoder (VAE-B)
## Explainable Hybrid Quantum-Classical NIDS

**Purpose:** Compress 140-feature Stage 2 output (near-zero variance features removed) into 8-dimensional latent vectors bounded to [0, π] for Stage 4 VQC qubit rotation angles.

**Model:** VAE-B — trained on Stage 2 data **without** near-zero variance features (140 features).

**Pipeline Position:**
```
Stage 2 (140 features, [0,1]) → Stage 3 VAE-B (8D, [0,π]) → Stage 4 VQC (8 qubits)
```

**Key Design Decisions:**
- Sentinel-masked ELBO loss (reconstructs only real features per sample)
- Quantum-safe bottleneck: `sigmoid(μ) × π` for gradient-preserving angle bounding
- KL annealing (0 → β over 10 epochs) to prevent posterior collapse
- GELU activations to avoid dying neurons with sentinel-rich inputs

## Cell 1 — Environment Setup

In [15]:
# ============================================================
# Cell 1: Environment Setup
# ============================================================
import sys
import os
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Reproducibility
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE);
np.random.seed(RANDOM_STATE);
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Python:  {sys.version}')
print(f'PyTorch: {torch.__version__}')
print(f'Device:  {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Python:  3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]
PyTorch: 2.9.0+cpu
Device:  cpu


## Cell 2 — Mount Google Drive & Set Paths

In [16]:
# ============================================================
# Cell 2: Mount Drive & Configure Paths
# ============================================================

# --- UPDATE THIS PATH to your Stage 2 output directory ---
STAGE2_DIR = '../DATA_stage_2_without_near_zero'
STAGE3_DIR = '../VAE_stage3_outputs'

os.makedirs(STAGE3_DIR, exist_ok=True)

# Verify Stage 2 artefacts exist
required_files = [
    'stage2_X_train.parquet',
    'stage2_X_test.parquet',
    'stage2_sentinel_mask_train.parquet',
    'stage2_sentinel_mask_test.parquet',
    'stage2_y_train.parquet',
    'stage2_y_test.parquet',
    'stage2_feature_names.json',
]

for f in required_files:
    path = os.path.join(STAGE2_DIR, f)
    assert os.path.exists(path), f'MISSING: {path}'
    size_mb = os.path.getsize(path) / 1e6
    print(f'  ✓ {f} ({size_mb:.1f} MB)')

print(f'\nStage 2 input:  {STAGE2_DIR}')
print(f'Stage 3 output: {STAGE3_DIR}')

  ✓ stage2_X_train.parquet (295.2 MB)
  ✓ stage2_X_test.parquet (72.4 MB)
  ✓ stage2_sentinel_mask_train.parquet (32.1 MB)
  ✓ stage2_sentinel_mask_test.parquet (7.8 MB)
  ✓ stage2_y_train.parquet (0.7 MB)
  ✓ stage2_y_test.parquet (0.2 MB)
  ✓ stage2_feature_names.json (0.0 MB)

Stage 2 input:  ../DATA_stage_2_without_near_zero
Stage 3 output: ../VAE_stage3_outputs


## Cell 3 — Load Stage 2 Contract

In [17]:
# ============================================================
# Cell 3: Load Stage 2 Contract
# ============================================================
with open(os.path.join(STAGE2_DIR, 'stage2_feature_names.json')) as f:
    s2_meta = json.load(f)

INPUT_DIM     = s2_meta['n_features']       # 140 (without near-zero variance)
QUANTUM_DIM   = s2_meta['quantum_dim']       # 8 — fixed, matches VQC qubit count
FEATURE_NAMES = s2_meta['feature_names']     # ordered list

assert QUANTUM_DIM == 8, f'QUANTUM_DIM must be 8 to match VQC qubit count, got {QUANTUM_DIM}'

print(f'INPUT_DIM   = {INPUT_DIM}')
print(f'QUANTUM_DIM = {QUANTUM_DIM}')
print(f'Features:     {FEATURE_NAMES[:5]} ... ({len(FEATURE_NAMES)} total)')

INPUT_DIM   = 140
QUANTUM_DIM = 8
Features:     ['ACK Flag Count', 'Active Max', 'Active Mean', 'Active Min', 'Active Std'] ... (140 total)


## Cell 4 — Streaming Dataset Class

In [18]:
# ============================================================
# Cell 4: Sentinel-Aware Parquet Dataset
# ============================================================

def _read_parquet_chunked(path, out_dtype=np.float32):
    """
    Read a Parquet file in row groups to avoid a single large malloc.
    Allocates one row group (~100K rows) at a time, then concatenates.
    Passing out_dtype=np.float16 halves peak RAM for the feature matrix.
    """
    import pyarrow.parquet as pq
    pf = pq.ParquetFile(path)
    chunks = []
    for i in range(pf.metadata.num_row_groups):
        tbl = pf.read_row_group(i)
        # to_pydict() returns {col: list}; column_stack stays in float64 briefly
        arr = np.column_stack([np.asarray(tbl.column(c), dtype=out_dtype)
                               for c in tbl.schema.names])
        chunks.append(arr)
    return np.concatenate(chunks, axis=0)


class SentinelAwareParquetDataset(Dataset):
    """
    Loads Stage 2 Parquet files into RAM using row-group streaming to avoid
    a single large malloc.  Feature matrix is stored as float16 (~660 MB
    instead of 1.32 GB) and cast to float32 inside __getitem__ so the
    training loop receives the expected dtype without additional RAM overhead.
    """
    def __init__(self, x_path, mask_path, label_path=None):
        print(f'Loading {os.path.basename(x_path)} ...')

        # Read features as float16 — halves RAM (P-04 fix)
        X_np   = _read_parquet_chunked(x_path,   out_dtype=np.float16)
        mask_np = _read_parquet_chunked(mask_path, out_dtype=np.uint8)

        self.X    = torch.from_numpy(X_np)                     # float16
        self.mask = torch.from_numpy(mask_np).bool()           # bool
        self.n_features = self.X.shape[1]

        if label_path:
            y_df  = pd.read_parquet(label_path)
            y_col = 'y' if 'y' in y_df.columns else y_df.columns[0]
            self.y = torch.tensor(y_df[y_col].values, dtype=torch.long)
        else:
            self.y = None

        ram_mb = (self.X.element_size() * self.X.nelement() +
                  self.mask.element_size() * self.mask.nelement()) / 1e6
        print(f'  X shape:    {self.X.shape}  (float16, {self.X.element_size()*self.X.nelement()/1e6:.0f} MB)')
        print(f'  mask shape: {self.mask.shape}')
        print(f'  Total RAM:  ~{ram_mb:.0f} MB')
        if self.y is not None:
            print(f'  y shape:    {self.y.shape}')
            labels, counts = torch.unique(self.y, return_counts=True)
            for lbl, cnt in zip(labels.tolist(), counts.tolist()):
                print(f'    Class {lbl}: {cnt:>10,}')

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # Cast float16 → float32 here (per-batch cost, no extra RAM at rest)
        x = self.X[idx].float()
        if self.y is not None:
            return x, self.mask[idx], self.y[idx]
        return x, self.mask[idx]

## Cell 5 — Build Datasets & DataLoaders

In [19]:
# ============================================================
# Cell 5: Build Datasets & DataLoaders
# ============================================================
print('=== TRAINING SET ===')
ds_train = SentinelAwareParquetDataset(
    x_path    = os.path.join(STAGE2_DIR, 'stage2_X_train.parquet'),
    mask_path = os.path.join(STAGE2_DIR, 'stage2_sentinel_mask_train.parquet'),
    label_path= os.path.join(STAGE2_DIR, 'stage2_y_train.parquet'),
)

print('\n=== TEST SET ===')
ds_test = SentinelAwareParquetDataset(
    x_path    = os.path.join(STAGE2_DIR, 'stage2_X_test.parquet'),
    mask_path = os.path.join(STAGE2_DIR, 'stage2_sentinel_mask_test.parquet'),
    label_path= os.path.join(STAGE2_DIR, 'stage2_y_test.parquet'),
)

assert ds_train.n_features == INPUT_DIM, \
    f'Train features {ds_train.n_features} != expected {INPUT_DIM}'
assert ds_test.n_features == INPUT_DIM, \
    f'Test features {ds_test.n_features} != expected {INPUT_DIM}'

BATCH_SIZE_TRAIN = 512
BATCH_SIZE_VAL   = 1024

# num_workers=0: avoids Windows multiprocessing pickle errors with tensors.
# pin_memory: only beneficial with CUDA; disable on CPU to save RAM.
_num_workers = 0
_pin_memory  = DEVICE.type == 'cuda'

train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE_TRAIN,
                          shuffle=True,  num_workers=_num_workers,
                          pin_memory=_pin_memory)
val_loader   = DataLoader(ds_test,  batch_size=BATCH_SIZE_VAL,
                          shuffle=False, num_workers=_num_workers,
                          pin_memory=_pin_memory)

print(f'\nTrain batches: {len(train_loader)} (batch_size={BATCH_SIZE_TRAIN})')
print(f'Val batches:   {len(val_loader)} (batch_size={BATCH_SIZE_VAL})')
print(f'num_workers:   {_num_workers}  pin_memory: {_pin_memory}')

=== TRAINING SET ===
Loading stage2_X_train.parquet ...
  X shape:    torch.Size([2367588, 140])  (float16, 663 MB)
  mask shape: torch.Size([2367588, 140])
  Total RAM:  ~994 MB
  y shape:    torch.Size([2367588])
    Class 0:  1,813,161
    Class 1:    312,881
    Class 2:    116,546
    Class 3:    100,000
    Class 4:     25,000

=== TEST SET ===
Loading stage2_X_test.parquet ...
  X shape:    torch.Size([573807, 140])  (float16, 161 MB)
  mask shape: torch.Size([573807, 140])
  Total RAM:  ~241 MB
  y shape:    torch.Size([573807])
    Class 0:    453,290
    Class 1:     78,221
    Class 2:     29,137
    Class 3:     11,966
    Class 4:      1,193

Train batches: 4625 (batch_size=512)
Val batches:   561 (batch_size=1024)
num_workers:   0  pin_memory: False


## Cell 6 — VAE Architecture

In [20]:
# ============================================================
# Cell 6: VAE Architecture
# ============================================================

def reparameterise(mu, log_var):
    """Standard reparameterisation trick for VAE training."""
    std = torch.exp(0.5 * log_var)
    eps = torch.randn_like(std)     # eps ~ N(0, I)
    return mu + eps * std            # z ~ N(mu, sigma^2)


def to_quantum_angles(mu):
    """
    Convert encoder mean output to qubit rotation angles.
    Output guaranteed in (0, pi) — open interval, never exactly 0 or pi.
    Use ONLY at inference time (after training), not during training.
    """
    return torch.sigmoid(mu) * torch.pi


class SentinelAwareVAE(nn.Module):
    """
    VAE with sentinel-aware reconstruction loss support.
    Architecture: INPUT_DIM -> 256 -> 128 -> 64 -> (mu:8, logvar:8)
    Decoder mirrors the encoder.
    """
    def __init__(self, input_dim, latent_dim=8, hidden_dims=None):
        super().__init__()
        self.input_dim  = input_dim
        self.latent_dim = latent_dim
        self.cond_dim   = 0  # Standard VAE (not conditional)

        if hidden_dims is None:
            hidden_dims = [256, 128, 64]

        # ── Encoder ─────────────────────────────────────────
        enc_layers = []
        in_d = input_dim
        for h in hidden_dims:
            enc_layers += [
                nn.Linear(in_d, h),
                nn.BatchNorm1d(h),
                nn.GELU(),
                nn.Dropout(0.1),
            ]
            in_d = h
        self.encoder = nn.Sequential(*enc_layers)

        # mu and log_var heads — no BatchNorm, no activation
        self.fc_mu      = nn.Linear(in_d, latent_dim)
        self.fc_log_var = nn.Linear(in_d, latent_dim)

        # ── Decoder ─────────────────────────────────────────
        dec_layers = []
        rev_dims = list(reversed(hidden_dims))
        in_d = latent_dim
        for h in rev_dims:
            dec_layers += [
                nn.Linear(in_d, h),
                nn.BatchNorm1d(h),
                nn.GELU(),
            ]
            in_d = h
        dec_layers.append(nn.Linear(in_d, input_dim))
        dec_layers.append(nn.Sigmoid())   # output in [0, 1]
        self.decoder = nn.Sequential(*dec_layers)

    def encode(self, x):
        h  = self.encoder(x)
        mu = self.fc_mu(h)
        lv = self.fc_log_var(h)
        # Clamp log_var to prevent NaN (Problem P-05 / P-07)
        lv = torch.clamp(lv, min=-10, max=10)
        return mu, lv

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = reparameterise(mu, log_var)   # sampled z for decoder
        x_recon = self.decode(z)
        return x_recon, mu, log_var

    def get_quantum_angles(self, x):
        """Inference only — returns deterministic angle vector in [0, pi]."""
        self.eval()
        with torch.no_grad():
            mu, _ = self.encode(x)
            return to_quantum_angles(mu)


# Print architecture summary
model_test = SentinelAwareVAE(input_dim=INPUT_DIM, latent_dim=QUANTUM_DIM)
total_params = sum(p.numel() for p in model_test.parameters())
trainable_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f'VAE-B Architecture: {INPUT_DIM} → 256 → 128 → 64 → 8 (latent)')
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
del model_test

VAE-B Architecture: 140 → 256 → 128 → 64 → 8 (latent)
Total parameters:     157,980
Trainable parameters: 157,980


## Cell 7 — Loss Function & KL Annealer

In [21]:
# ============================================================
# Cell 7: Sentinel-Masked ELBO Loss & KL Annealer
# ============================================================

def vae_loss(x, x_recon, mu, log_var, sentinel_mask, beta=1.0):
    """
    Sentinel-masked ELBO loss.

    Reconstruction loss is computed only over real (non-sentinel) features,
    normalised by the number of real features per sample to ensure equal
    contribution from all three datasets regardless of sentinel rate.

    Parameters
    ----------
    x             : (batch, F) float32 — original input
    x_recon       : (batch, F) float32 — decoder output
    mu            : (batch, 8) float32 — encoder mean
    log_var       : (batch, 8) float32 — encoder log-variance
    sentinel_mask : (batch, F) bool    — True = absent feature
    beta          : float — KL weight (beta-VAE if > 1.0)

    Returns
    -------
    total_loss, recon_loss, kl_loss  (all scalars)
    """
    # Weight: 1.0 for real features, 0.0 for sentinel positions
    weight = (~sentinel_mask).float()                # (batch, F)

    # Per-element squared error, zeroed at sentinel positions
    sq_err = weight * (x - x_recon).pow(2)           # (batch, F)

    # Normalise by number of REAL features per sample
    real_count = weight.sum(dim=1).clamp(min=1)      # (batch,)
    recon_loss = (sq_err.sum(dim=1) / real_count).mean()

    # KL divergence: D_KL(N(mu, sigma^2) || N(0,1))
    kl_loss = -0.5 * (1 + log_var - mu.pow(2) - log_var.exp())
    kl_loss = kl_loss.sum(dim=1).mean()

    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss, kl_loss


class KLAnnealer:
    """
    Linearly anneals beta from 0 -> beta_target over warmup_epochs,
    then holds at beta_target.
    Prevents posterior collapse at training start (Problem P-01).
    """
    def __init__(self, beta_target=1.0, warmup_epochs=10):
        self.beta_target   = beta_target
        self.warmup_epochs = warmup_epochs

    def get_beta(self, epoch):
        if epoch >= self.warmup_epochs:
            return self.beta_target
        return self.beta_target * (epoch / self.warmup_epochs)


# ── Smoke test ──────────────────────────────────────────────
print('Loss function smoke test:')
_x    = torch.rand(4, INPUT_DIM)
_xr   = torch.rand(4, INPUT_DIM)
_mu   = torch.randn(4, 8)
_lv   = torch.randn(4, 8)
_mask = torch.zeros(4, INPUT_DIM, dtype=torch.bool)
_mask[:, -20:] = True   # simulate some sentinel positions
_total, _recon, _kl = vae_loss(_x, _xr, _mu, _lv, _mask, beta=1.0)
print(f'  total={_total.item():.4f}  recon={_recon.item():.4f}  kl={_kl.item():.4f}')
assert all(torch.isfinite(t) for t in [_total, _recon, _kl]), 'Smoke test FAILED: non-finite loss'
print('  Smoke test PASSED: all losses are finite scalars')

Loss function smoke test:
  total=5.5907  recon=0.1621  kl=5.4287
  Smoke test PASSED: all losses are finite scalars


## Cell 8 — Training Configuration

In [22]:
# ============================================================
# Cell 8: Training Configuration
# ============================================================
CONFIG = {
    'model_name':    'vae_b',
    'input_dim':     INPUT_DIM,         # 140
    'latent_dim':    QUANTUM_DIM,       # 8
    'hidden_dims':   [256, 128, 64],
    'lr':            1e-3,
    'weight_decay':  1e-4,
    'batch_size':    BATCH_SIZE_TRAIN,  # 512
    'max_epochs':    100,
    'patience':      15,               # early stopping patience
    'beta_target':   1.0,              # KL weight
    'warmup_epochs': 10,               # KL annealing warmup
    'grad_clip':     1.0,              # gradient clipping max_norm
    'scheduler_factor':   0.5,
    'scheduler_patience': 5,
}

print('=== VAE-B Training Configuration ===')
for k, v in CONFIG.items():
    print(f'  {k:25s}: {v}')

=== VAE-B Training Configuration ===
  model_name               : vae_b
  input_dim                : 140
  latent_dim               : 8
  hidden_dims              : [256, 128, 64]
  lr                       : 0.001
  weight_decay             : 0.0001
  batch_size               : 512
  max_epochs               : 100
  patience                 : 15
  beta_target              : 1.0
  warmup_epochs            : 10
  grad_clip                : 1.0
  scheduler_factor         : 0.5
  scheduler_patience       : 5


## Cell 9 — Instantiate VAE-B Model

In [23]:
# ============================================================
# Cell 9: Instantiate VAE-B
# ============================================================
model = SentinelAwareVAE(
    input_dim   = CONFIG['input_dim'],
    latent_dim  = CONFIG['latent_dim'],
    hidden_dims = CONFIG['hidden_dims'],
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'VAE-B on {DEVICE}')
print(f'Parameters: {total_params:,}')

# Forward pass sanity check with one batch
sample_x, sample_mask, sample_y = next(iter(train_loader))
sample_x = sample_x.to(DEVICE)
with torch.no_grad():
    x_recon, mu, log_var = model(sample_x)
    angles = to_quantum_angles(mu)

print(f'\nForward pass check:')
print(f'  Input:      {sample_x.shape}')
print(f'  Recon:      {x_recon.shape}')
print(f'  mu:         {mu.shape}  range=[{mu.min():.3f}, {mu.max():.3f}]')
print(f'  log_var:    {log_var.shape}  range=[{log_var.min():.3f}, {log_var.max():.3f}]')
print(f'  angles:     {angles.shape}  range=[{angles.min():.4f}, {angles.max():.4f}]')
print(f'  angles in [0, pi]: {(angles >= 0).all() and (angles <= torch.pi).all()}')

VAE-B on cpu
Parameters: 157,980

Forward pass check:
  Input:      torch.Size([512, 140])
  Recon:      torch.Size([512, 140])
  mu:         torch.Size([512, 8])  range=[-2.078, 3.058]
  log_var:    torch.Size([512, 8])  range=[-2.266, 2.469]
  angles:     torch.Size([512, 8])  range=[0.3494, 3.0007]
  angles in [0, pi]: True


## Cell 10 — Training Loop

In [24]:
# ============================================================
# Cell 10: Training Loop
# ============================================================

def train_vae(model, train_loader, val_loader, config, device):
    """Complete training loop with KL annealing, early stopping, and gradient clipping."""
    model_name = config['model_name']

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config['weight_decay'],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min',
        factor=config['scheduler_factor'],
        patience=config['scheduler_patience'],
        # verbose= removed: deprecated in PyTorch 2.x
    )
    annealer = KLAnnealer(
        beta_target=config['beta_target'],
        warmup_epochs=config['warmup_epochs'],
    )

    best_val_recon = float('inf')
    best_epoch     = 0
    patience_ctr   = 0
    history        = []
    checkpoint_path = os.path.join(STAGE3_DIR, f'{model_name}_best.pt')

    print(f'\n{"="*60}')
    print(f'  Training {model_name.upper()} — max {config["max_epochs"]} epochs')
    print(f'  Early stopping patience: {config["patience"]}')
    print(f'  KL annealing: 0 → {config["beta_target"]} over {config["warmup_epochs"]} epochs')
    print(f'{"="*60}\n')

    t_start = time.time()

    for epoch in range(1, config['max_epochs'] + 1):
        beta = annealer.get_beta(epoch - 1)

        # ── Training phase ──────────────────────────────────
        model.train()
        tr_total = tr_recon = tr_kl = 0.0
        n_batches = 0

        for batch in train_loader:
            x    = batch[0].to(device)
            mask = batch[1].to(device)

            x_recon, mu, log_var = model(x)
            loss, recon, kl = vae_loss(x, x_recon, mu, log_var, mask, beta)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),
                                           max_norm=config['grad_clip'])
            optimizer.step()

            tr_total += loss.item()
            tr_recon += recon.item()
            tr_kl    += kl.item()
            n_batches += 1

        tr_total /= n_batches
        tr_recon /= n_batches
        tr_kl    /= n_batches

        # ── Validation phase ────────────────────────────────
        model.eval()
        val_total = val_recon = val_kl = 0.0
        n_val = 0

        with torch.no_grad():
            for batch in val_loader:
                x    = batch[0].to(device)
                mask = batch[1].to(device)

                x_recon, mu, log_var = model(x)
                loss, recon, kl = vae_loss(x, x_recon, mu, log_var, mask, beta)

                val_total += loss.item()
                val_recon += recon.item()
                val_kl    += kl.item()
                n_val += 1

        val_total /= n_val
        val_recon /= n_val
        val_kl    /= n_val

        # ── LR scheduler step ──────────────────────────────
        prev_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_recon)
        current_lr = optimizer.param_groups[0]['lr']
        if current_lr < prev_lr:
            print(f'  [LR] Reduced: {prev_lr:.2e} → {current_lr:.2e}')

        # ── Logging ────────────────────────────────────────
        history.append({
            'epoch': epoch, 'beta': beta, 'lr': current_lr,
            'tr_total': tr_total, 'tr_recon': tr_recon, 'tr_kl': tr_kl,
            'val_total': val_total, 'val_recon': val_recon, 'val_kl': val_kl,
        })

        if epoch % 5 == 0 or epoch == 1:
            elapsed = time.time() - t_start
            print(f'[{model_name}] E{epoch:03d} β={beta:.2f} lr={current_lr:.1e} | '
                  f'tr_recon={tr_recon:.4f} tr_kl={tr_kl:.4f} | '
                  f'val_recon={val_recon:.4f} val_kl={val_kl:.4f} | '
                  f'{elapsed:.0f}s')

        # ── Early stopping & checkpoint ────────────────────
        if val_recon < best_val_recon - 1e-6:
            best_val_recon = val_recon
            best_epoch     = epoch
            patience_ctr   = 0
            torch.save({
                'epoch':           epoch,
                'model_state':     model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'val_recon':       val_recon,
                'config':          config,
            }, checkpoint_path)
        else:
            patience_ctr += 1
            if patience_ctr >= config['patience']:
                print(f'\n[{model_name}] Early stop at epoch {epoch}. '
                      f'Best: epoch {best_epoch}, val_recon={best_val_recon:.6f}')
                break

    total_time = time.time() - t_start
    print(f'\n[{model_name}] Training complete in {total_time:.0f}s ({total_time/60:.1f} min)')
    print(f'[{model_name}] Best epoch: {best_epoch}, val_recon: {best_val_recon:.6f}')
    print(f'[{model_name}] Checkpoint saved: {checkpoint_path}')

    return history

## Cell 11 — Train VAE-B

In [25]:
# ============================================================
# Cell 11: Train VAE-B
# ============================================================
history = train_vae(model, train_loader, val_loader, CONFIG, DEVICE)

# Save training history
history_path = os.path.join(STAGE3_DIR, f'{CONFIG["model_name"]}_training_history.json')
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f'Training history saved: {history_path}')


  Training VAE_B — max 100 epochs
  Early stopping patience: 15
  KL annealing: 0 → 1.0 over 10 epochs

[vae_b] E001 β=0.00 lr=1.0e-03 | tr_recon=0.0014 tr_kl=43.1648 | val_recon=0.0002 val_kl=54.7071 | 283s
[vae_b] E005 β=0.40 lr=1.0e-03 | tr_recon=0.0259 tr_kl=0.0000 | val_recon=0.0253 val_kl=0.0000 | 1227s
  [LR] Reduced: 1.00e-03 → 5.00e-04
[vae_b] E010 β=0.90 lr=5.0e-04 | tr_recon=0.0259 tr_kl=0.0000 | val_recon=0.0253 val_kl=0.0000 | 2427s
  [LR] Reduced: 5.00e-04 → 2.50e-04
[vae_b] E015 β=1.00 lr=2.5e-04 | tr_recon=0.0259 tr_kl=0.0000 | val_recon=0.0253 val_kl=0.0000 | 4116s

[vae_b] Early stop at epoch 16. Best: epoch 1, val_recon=0.000185

[vae_b] Training complete in 4353s (72.5 min)
[vae_b] Best epoch: 1, val_recon: 0.000185
[vae_b] Checkpoint saved: ../VAE_stage3_outputs\vae_b_best.pt
Training history saved: ../VAE_stage3_outputs\vae_b_training_history.json


## Cell 12 — Training Loss Curves

In [27]:
# ============================================================
# Cell 12: Plot Training Curves
# ============================================================
import matplotlib.pyplot as plt
epochs     = [h['epoch'] for h in history]
tr_recon   = [h['tr_recon'] for h in history]
val_recon  = [h['val_recon'] for h in history]
tr_kl      = [h['tr_kl'] for h in history]
val_kl     = [h['val_kl'] for h in history]
betas      = [h['beta'] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Reconstruction loss
axes[0].plot(epochs, tr_recon, label='Train Recon', color='tab:blue')
axes[0].plot(epochs, val_recon, label='Val Recon', color='tab:orange')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Masked Reconstruction Loss')
axes[0].set_title('Reconstruction Loss (Sentinel-Masked)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# KL divergence
axes[1].plot(epochs, tr_kl, label='Train KL', color='tab:blue')
axes[1].plot(epochs, val_kl, label='Val KL', color='tab:orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('KL Divergence')
axes[1].set_title('KL Divergence')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Beta annealing
axes[2].plot(epochs, betas, color='tab:green')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Beta')
axes[2].set_title('KL Annealing Schedule')
axes[2].grid(True, alpha=0.3)

fig.suptitle('VAE-B Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(STAGE3_DIR, 'vae_b_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved.')

ModuleNotFoundError: No module named 'matplotlib'

## Cell 13 — Load Best Checkpoint

In [ ]:
# ============================================================
# Cell 13: Load Best Checkpoint
# ============================================================
checkpoint_path = os.path.join(STAGE3_DIR, f'{CONFIG["model_name"]}_best.pt')
checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

model.load_state_dict(checkpoint['model_state'])
model.eval()

print(f'Loaded best checkpoint:')
print(f'  Epoch:     {checkpoint["epoch"]}')
print(f'  Val Recon: {checkpoint["val_recon"]:.6f}')

## Cell 14 — Extract Latent Vectors (Quantum Angles)

In [ ]:
# ============================================================
# Cell 14: Extract & Save Latent Vectors
# ============================================================

def extract_and_save_latent(model, data_loader, device, output_path):
    """
    Extract quantum angle vectors (sigmoid(mu) * pi) for all samples.
    Validates range [0, pi] before saving to Parquet.
    """
    col_names = [f'z_{i}' for i in range(8)]
    model.eval()
    all_angles = []

    with torch.no_grad():
        for batch in data_loader:
            x = batch[0].to(device)
            mu, _ = model.encode(x)
            angles = to_quantum_angles(mu).cpu()
            all_angles.append(angles)

    z_all = torch.cat(all_angles, dim=0)

    # Validate before save
    assert z_all.shape[1] == 8, f'Expected 8 latent dims, got {z_all.shape[1]}'
    assert (z_all >= 0).all(), f'Values below 0 found: min={z_all.min():.6f}'
    assert (z_all <= torch.pi).all(), f'Values above pi found: max={z_all.max():.6f}'
    assert not torch.isnan(z_all).any(), 'NaN values found in latent vectors'
    assert not torch.isinf(z_all).any(), 'Inf values found in latent vectors'

    df = pd.DataFrame(z_all.numpy(), columns=col_names)
    df.to_parquet(output_path, index=False)

    print(f'Saved: {output_path}')
    print(f'  Shape: {z_all.shape}')
    print(f'  Range: [{z_all.min():.4f}, {z_all.max():.4f}]')
    print(f'  Per-dim std: {[f"{s:.4f}" for s in z_all.std(dim=0).tolist()]}')

    return z_all


# Non-shuffled loaders for extraction (preserves sample order for label alignment)
train_extract_loader = DataLoader(ds_train, batch_size=BATCH_SIZE_VAL,
                                  shuffle=False, num_workers=0,
                                  pin_memory=_pin_memory)
test_extract_loader  = DataLoader(ds_test,  batch_size=BATCH_SIZE_VAL,
                                  shuffle=False, num_workers=0,
                                  pin_memory=_pin_memory)

print('=== Extracting z_train ===')
z_train = extract_and_save_latent(
    model, train_extract_loader, DEVICE,
    os.path.join(STAGE3_DIR, 'vae_b_z_train.parquet'),
)

print('\n=== Extracting z_test ===')
z_test = extract_and_save_latent(
    model, test_extract_loader, DEVICE,
    os.path.join(STAGE3_DIR, 'vae_b_z_test.parquet'),
)

## Cell 15 — 8-Point Validation Protocol

In [ ]:
# ============================================================
# Cell 15: Stage 3 Output Validation (8-Point Gate)
# ============================================================

def validate_stage3_output(z_train, z_test, y_train, y_test, model_name):
    """
    8-point validation gate. ALL assertions must pass before
    z_train / z_test are considered valid for Stage 4.
    """
    class_names = {0: 'NORMALL', 1: 'DoSD', 2: 'PROBE', 3: 'EXPLOIT', 4: 'MALWARE'}
    print(f'\n{"="*55}')
    print(f'  STAGE 3 OUTPUT VALIDATION: {model_name}')
    print(f'{"="*55}')

    # 1. Shape consistency
    assert z_train.shape[1] == 8, f'z_train latent dim = {z_train.shape[1]}, expected 8'
    assert z_test.shape[1] == 8,  f'z_test latent dim = {z_test.shape[1]}, expected 8'
    assert z_train.shape[0] == len(y_train), \
        f'z_train rows {z_train.shape[0]} != y_train {len(y_train)}'
    assert z_test.shape[0] == len(y_test), \
        f'z_test rows {z_test.shape[0]} != y_test {len(y_test)}'
    print('  [1/8] Shape consistency:         PASS')

    # 2. Range [0, pi]
    eps = 1e-6
    assert float(z_train.min()) >= -eps, f'z_train min = {z_train.min():.6f}, below 0'
    assert float(z_train.max()) <= torch.pi + eps, f'z_train max = {z_train.max():.6f}, above pi'
    assert float(z_test.min()) >= -eps, f'z_test min = {z_test.min():.6f}, below 0'
    assert float(z_test.max()) <= torch.pi + eps, f'z_test max = {z_test.max():.6f}, above pi'
    print('  [2/8] Range [0, pi]:             PASS')

    # 3. No NaN
    assert not torch.isnan(z_train).any(), 'NaN in z_train'
    assert not torch.isnan(z_test).any(),  'NaN in z_test'
    print('  [3/8] No NaN:                    PASS')

    # 4. No Inf
    assert not torch.isinf(z_train).any(), 'Inf in z_train'
    assert not torch.isinf(z_test).any(),  'Inf in z_test'
    print('  [4/8] No Inf:                    PASS')

    # 5. Per-dimension std (angle diversity)
    dim_std = z_train.std(dim=0)
    collapsed_dims = (dim_std < 0.05).nonzero(as_tuple=True)[0].tolist()
    if collapsed_dims:
        print(f'  [5/8] WARNING: dims {collapsed_dims} have std < 0.05 — near-collapsed')
        print(f'         Stds: {[f"{s:.4f}" for s in dim_std.tolist()]}')
    else:
        print(f'  [5/8] Angle diversity:           PASS  (min_std={dim_std.min():.4f})')

    # 6. Class separation in latent space
    classes = torch.unique(y_train)
    centroids = {}
    for cls in classes:
        centroids[int(cls)] = z_train[y_train == cls].mean(dim=0)

    min_dist = float('inf')
    min_pair = None
    for i in range(len(classes)):
        for j in range(i + 1, len(classes)):
            ci, cj = int(classes[i]), int(classes[j])
            d = (centroids[ci] - centroids[cj]).norm().item()
            if d < min_dist:
                min_dist = d
                min_pair = (class_names[ci], class_names[cj])
    print(f'  [6/8] Min inter-class dist:      {min_dist:.4f}  ({min_pair[0]} vs {min_pair[1]})')
    if min_dist < 0.1:
        print('         WARNING: Very low separation. Consider cVAE or auxiliary loss.')

    # 7. Test set angle diversity
    test_std = z_test.std(dim=0)
    const_test = (test_std < 1e-4).nonzero(as_tuple=True)[0].tolist()
    if const_test:
        print(f'  [7/8] WARNING: z_test dims {const_test} are near-constant')
    else:
        print(f'  [7/8] Test angle diversity:      PASS  (min_std={test_std.min():.4f})')

    # 8. Label distribution — all 5 classes present
    expected_classes = {0, 1, 2, 3, 4}
    found_classes = set(torch.unique(y_test).tolist())
    assert found_classes == expected_classes, \
        f'Test set missing classes: {expected_classes - found_classes}'
    print('  [8/8] All 5 classes in test set: PASS')

    print(f'{"="*55}')
    print(f'  ALL VALIDATION CHECKS COMPLETE FOR {model_name}')
    print(f'{"="*55}')

    return {
        'dim_std': dim_std.tolist(),
        'min_class_dist': min_dist,
        'min_class_pair': list(min_pair),
        'collapsed_dims': collapsed_dims,
        'const_test_dims': const_test,
        'centroids': {class_names[k]: v.tolist() for k, v in centroids.items()},
    }


# Run validation
y_train_tensor = ds_train.y
y_test_tensor  = ds_test.y

validation_results = validate_stage3_output(
    z_train, z_test, y_train_tensor, y_test_tensor,
    model_name='VAE-B',
)

## Cell 16 — Latent Space Analysis

In [ ]:
# ============================================================
# Cell 16: Latent Space Analysis
# ============================================================
import matplotlib.pyplot as plt

class_names = {0: 'NORMALL', 1: 'DoSD', 2: 'PROBE', 3: 'EXPLOIT', 4: 'MALWARE'}
class_colors = {0: '#2196F3', 1: '#F44336', 2: '#FF9800', 3: '#9C27B0', 4: '#4CAF50'}

# ── 16a: Per-dimension angle distribution histograms ─────────
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for dim in range(8):
    ax = axes[dim]
    for cls_id in range(5):
        cls_mask = y_train_tensor == cls_id
        vals = z_train[cls_mask, dim].numpy()
        ax.hist(vals, bins=50, alpha=0.5, label=class_names[cls_id],
                color=class_colors[cls_id], density=True)
    ax.set_title(f'z_{dim}  (std={z_train[:, dim].std():.3f})')
    ax.set_xlabel('Angle (radians)')
    ax.axvline(x=np.pi/2, color='gray', linestyle='--', alpha=0.5)
    if dim == 0:
        ax.legend(fontsize=7)

fig.suptitle('VAE-B: Per-Dimension Angle Distributions by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(STAGE3_DIR, 'vae_b_angle_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()


# ── 16b: Inter-class centroid distance matrix ────────────────
centroids = validation_results['centroids']
cls_order = ['NORMALL', 'DoSD', 'PROBE', 'EXPLOIT', 'MALWARE']
n_cls = len(cls_order)
dist_matrix = np.zeros((n_cls, n_cls))

for i in range(n_cls):
    for j in range(n_cls):
        ci = np.array(centroids[cls_order[i]])
        cj = np.array(centroids[cls_order[j]])
        dist_matrix[i, j] = np.linalg.norm(ci - cj)

print('\n=== Inter-Class Centroid Distance Matrix (8D Euclidean) ===')
print(f'{"":>10s}', end='')
for name in cls_order:
    print(f'{name:>10s}', end='')
print()
for i, name_i in enumerate(cls_order):
    print(f'{name_i:>10s}', end='')
    for j in range(n_cls):
        print(f'{dist_matrix[i, j]:>10.4f}', end='')
    print()


# ── 16c: t-SNE visualisation on 5K stratified subset ────────
from sklearn.manifold import TSNE

N_TSNE = 5000
# Stratified sampling
indices = []
for cls_id in range(5):
    cls_indices = (y_train_tensor == cls_id).nonzero(as_tuple=True)[0]
    n_sample = min(N_TSNE // 5, len(cls_indices))
    perm = torch.randperm(len(cls_indices))[:n_sample]
    indices.append(cls_indices[perm])
indices = torch.cat(indices)

z_subset = z_train[indices].numpy()
y_subset = y_train_tensor[indices].numpy()

print(f'\nRunning t-SNE on {len(z_subset)} samples (stratified) ...')
tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=RANDOM_STATE)
z_2d = tsne.fit_transform(z_subset)

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
for cls_id in range(5):
    mask = y_subset == cls_id
    ax.scatter(z_2d[mask, 0], z_2d[mask, 1],
              c=class_colors[cls_id], label=class_names[cls_id],
              s=8, alpha=0.6)

ax.set_title('VAE-B: t-SNE of 8D Latent Space (5K stratified subset)', fontsize=13)
ax.set_xlabel('t-SNE dim 1')
ax.set_ylabel('t-SNE dim 2')
ax.legend(markerscale=3)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(STAGE3_DIR, 'vae_b_tsne.png'), dpi=150, bbox_inches='tight')
plt.show()
print('t-SNE plot saved.')

## Cell 17 — Per-Class Reconstruction Loss

In [ ]:
# ============================================================
# Cell 17: Per-Class Reconstruction Loss (Diagnostic)
# ============================================================
print('=== Per-Class Masked Reconstruction Loss (Validation Set) ===')

model.eval()
per_class_losses = {i: [] for i in range(5)}

with torch.no_grad():
    for batch in val_loader:
        x    = batch[0].to(DEVICE)
        mask = batch[1].to(DEVICE)
        y    = batch[2]

        x_recon, mu, log_var = model(x)

        weight = (~mask).float()
        sq_err = weight * (x - x_recon).pow(2)
        real_count = weight.sum(dim=1).clamp(min=1)
        per_sample_loss = (sq_err.sum(dim=1) / real_count).cpu()

        for cls_id in range(5):
            cls_mask = (y == cls_id)
            if cls_mask.any():
                per_class_losses[cls_id].append(per_sample_loss[cls_mask])

class_names_map = {0: 'NORMALL', 1: 'DoSD', 2: 'PROBE', 3: 'EXPLOIT', 4: 'MALWARE'}
per_class_summary = {}

for cls_id in range(5):
    losses = torch.cat(per_class_losses[cls_id])
    mean_l = losses.mean().item()
    std_l  = losses.std().item()
    per_class_summary[class_names_map[cls_id]] = {'mean': mean_l, 'std': std_l, 'n': len(losses)}
    print(f'  {class_names_map[cls_id]:>10s}:  mean={mean_l:.6f}  std={std_l:.6f}  n={len(losses):>8,}')

# Check for SMOTE degradation (Problem P-06)
normall_loss = per_class_summary['NORMALL']['mean']
for cls in ['EXPLOIT', 'MALWARE']:
    ratio = per_class_summary[cls]['mean'] / normall_loss if normall_loss > 0 else 0
    if ratio > 2.0:
        print(f'\n  WARNING: {cls} recon loss is {ratio:.1f}x NORMALL — '
              f'consider training on real samples only (Option A, Problem P-06)')

## Cell 18 — Save All Artefacts

In [ ]:
# ============================================================
# Cell 18: Save All Artefacts
# ============================================================

# ── Config JSON ────────────────────────────────────────────
config_save = CONFIG.copy()
config_save['best_epoch']     = checkpoint['epoch']
config_save['best_val_recon'] = checkpoint['val_recon']
config_save['epochs_trained'] = len(history)
config_save['feature_names']  = FEATURE_NAMES
config_save['device']         = str(DEVICE)

config_path = os.path.join(STAGE3_DIR, 'vae_b_config.json')
with open(config_path, 'w') as f:
    json.dump(config_save, f, indent=2)

# ── Latent stats JSON ──────────────────────────────────────
latent_stats = {
    'model_name': 'VAE-B',
    'input_dim': INPUT_DIM,
    'latent_dim': QUANTUM_DIM,
    'per_dim_mean': z_train.mean(dim=0).tolist(),
    'per_dim_std':  z_train.std(dim=0).tolist(),
    'per_dim_min':  z_train.min(dim=0).values.tolist(),
    'per_dim_max':  z_train.max(dim=0).values.tolist(),
    'class_centroids': validation_results['centroids'],
    'min_class_distance': validation_results['min_class_dist'],
    'min_class_pair':     validation_results['min_class_pair'],
    'collapsed_dims':     validation_results['collapsed_dims'],
    'z_train_shape': list(z_train.shape),
    'z_test_shape':  list(z_test.shape),
}

stats_path = os.path.join(STAGE3_DIR, 'vae_b_latent_stats.json')
with open(stats_path, 'w') as f:
    json.dump(latent_stats, f, indent=2)

# ── Model selection JSON (single-model run) ───────────────
selection = {
    'selected_model': 'VAE-B',
    'reason': 'VAE-B only (without near-zero variance features). VAE-A not trained in this run.',
    'vae_b_val_recon':  checkpoint['val_recon'],
    'vae_b_best_epoch': checkpoint['epoch'],
    'vae_b_input_dim':  INPUT_DIM,
    'z_train_path': 'vae_b_z_train.parquet',
    'z_test_path':  'vae_b_z_test.parquet',
}

selection_path = os.path.join(STAGE3_DIR, 'stage3_selected_model.json')
with open(selection_path, 'w') as f:
    json.dump(selection, f, indent=2)

# ── Verify all artefacts exist ─────────────────────────────
print('\n=== Stage 3 Output Artefacts ===')
expected_artefacts = [
    'vae_b_best.pt',
    'vae_b_training_history.json',
    'vae_b_config.json',
    'vae_b_latent_stats.json',
    'vae_b_z_train.parquet',
    'vae_b_z_test.parquet',
    'vae_b_training_curves.png',
    'vae_b_angle_distributions.png',
    'vae_b_tsne.png',
    'stage3_selected_model.json',
]

all_present = True
for fname in expected_artefacts:
    fpath = os.path.join(STAGE3_DIR, fname)
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'  OK  {fname} ({size_mb:.2f} MB)')
    else:
        print(f'  MISSING  {fname}')
        all_present = False

print()
if all_present:
    print('ALL ARTEFACTS SAVED SUCCESSFULLY')
    print('STAGE 3 COMPLETE — READY FOR STAGE 4 (VQC)')
else:
    print('WARNING: Some artefacts are missing. Check errors above.')

## Cell 19 — Summary Report

In [ ]:
# ============================================================
# Cell 19: Final Summary Report
# ============================================================
print('=' * 60)
print('  STAGE 3 — VAE-B TRAINING SUMMARY')
print('=' * 60)
print(f'  Model:              VAE-B (without near-zero variance features)')
print(f'  Input dimension:    {INPUT_DIM}')
print(f'  Latent dimension:   {QUANTUM_DIM}')
print(f'  Architecture:       {INPUT_DIM} -> 256 -> 128 -> 64 -> 8')
print(f'  Parameters:         {sum(p.numel() for p in model.parameters()):,}')
print(f'  Device:             {DEVICE}')
print(f'  Epochs trained:     {len(history)}')
print(f'  Best epoch:         {checkpoint["epoch"]}')
print(f'  Best val_recon:     {checkpoint["val_recon"]:.6f}')
print(f'  z_train shape:      {list(z_train.shape)}')
print(f'  z_test shape:       {list(z_test.shape)}')
print(f'  Angle range:        [{z_train.min():.4f}, {z_train.max():.4f}]')
print(f'  Per-dim std (mean): {z_train.std(dim=0).mean():.4f}')
print(f'  Min class distance: {validation_results["min_class_dist"]:.4f}')
print(f'  Collapsed dims:     {validation_results["collapsed_dims"]}')
print(f'  Output directory:   {STAGE3_DIR}')
print('=' * 60)
print('\n  Stage 4 should load:')
print(f'    - {STAGE3_DIR}/vae_b_z_train.parquet  (train angles)')
print(f'    - {STAGE3_DIR}/vae_b_z_test.parquet   (test angles)')
print(f'    - {STAGE2_DIR}/stage2_y_train.parquet (train labels)')
print(f'    - {STAGE2_DIR}/stage2_y_test.parquet  (test labels)')
print('=' * 60)